# Week 4 Day 4 - Exercise: browser tools, reading the sandbox, and a skill of my own

Exercise only. It reuses the web search tool, the `sandbox` folder and the model from the lab, set up in the cell below.

In [ ]:
# Setup shared with the exercise: imports, search tool, sandbox and model

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

load_dotenv(override=True)

search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

## Exercise, step 1: a second tool for the researcher

We add the Playwright MCP browser tools next to `search`. The browser tools are async, so we run the agent with `await ...ainvoke(...)`. The browser opens headed, so you can watch it work.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools")

In [ ]:
researcher_browser = create_deep_agent(
    model=model,
    tools=[search, *browser_tools],
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool. "
        "Use the search tool to find sources, then use the browser tools to open the most relevant page "
        "and read the real figures from it. Write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
brief_browser = """
Our company is planning to move its sales fleet to electric vehicles.
Find out roughly how many public EV charging ports there are in the US.
Search for a reliable source, then open that page in the browser and quote the figure from it.
Write a short markdown briefing with a heading, the figure, and the source URL to the file charging_browser.md.
"""

result = await researcher_browser.ainvoke({"messages": [{"role": "user", "content": brief_browser}]})
print(result["messages"][-1].content)

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)

## Exercise, step 2: read every file the agent wrote

The agent's filesystem is the `sandbox` folder. This cell walks it and prints each text file, so we can read the briefings, the comparison and anything else the agent saved for itself along the way. Binary files such as `.pptx` are only listed.

In [ ]:
from pathlib import Path

TEXT_SUFFIXES = {".md", ".txt", ".json", ".py"}

for path in sorted(Path(sandbox).rglob("*")):
    if not path.is_file():
        continue
    rel = path.relative_to(sandbox)
    print("=" * 70)
    print(f"{rel}  ({path.stat().st_size} bytes)")
    print("=" * 70)
    if path.suffix in TEXT_SUFFIXES:
        print(path.read_text())
    else:
        print("[binary file, not printed]")
    print()

## Exercise, step 3 (stretch): a skill of my own

A new skill, `sandbox/skills_midnight/midnight-slide/SKILL.md`, teaches the slide-maker a minimal midnight blue house style: a bold statement title, three short fragments and one confident sentence. The look itself lives in a second template, `midnight_slide_kit.py`, wrapped by a `create_midnight_slide` tool.

The skill sits in its own `skills_midnight` folder so that `skills=["/skills_midnight/"]` loads only this one and not Voltway's.

In [ ]:
from langchain_core.tools import tool
from midnight_slide_kit import build_midnight_slide

@tool
def create_midnight_slide(title: str, key_points: list[str], recommendation: str) -> str:
    """Create a one-slide PowerPoint in the midnight blue style, saved as midnight_fleet.pptx."""
    build_midnight_slide(title, key_points, recommendation, os.path.join(sandbox, "midnight_fleet.pptx"))
    return "Saved the slide to /midnight_fleet.pptx"

midnight_slide_maker = {
    "name": "midnight-slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck in the midnight blue style.",
    "system_prompt": "You turn research recommendations into slides, following your midnight-slide skill.",
    "tools": [create_midnight_slide],
    "skills": ["/skills_midnight/"],
}

midnight_presenter = create_deep_agent(
    model=model,
    subagents=[midnight_slide_maker],
    system_prompt="You prepare research for presentation by delegating to your midnight-slide-maker sub-agent.",
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
result = midnight_presenter.invoke({"messages": [{"role": "user", "content":
    "Read charging_browser.md and have a one-slide deck made of its recommendation."}]})
print(result["messages"][-1].content)

Open `sandbox/midnight_fleet.pptx` to see the slide.